In [1]:
import numpy as np
import pandas as pd

In [3]:
NOTIONAL_CONTRACTS = {
    "atm_call": 1,
    "atm_put": 1,
    "otm_call": 1,
    "otm_put": 1
}

In [4]:
def import_data():
    session_signals = pd.read_csv('data/session_signals.csv')
    options_df = pd.read_csv("data/options_df.csv")
    btc_prices = pd.read_csv('data/btc_prices.csv')
    btc_prices.rename(columns={'Unnamed: 0': 'time'}, inplace=True)
    return session_signals, options_df, btc_prices

In [13]:
def construct_trades(session_signals, options_df):
    positive_signals = session_signals[session_signals['signal']]
    trades_df = options_df[options_df['session_start'].isin(positive_signals['session_start'])].copy()

    trades_df['position'] = np.select(
        [
            trades_df['leg'].str.startswith('atm'),
            trades_df['leg'].str.startswith('otm')
        ],
        ['short', 'long'],
        default=None
    )

    trades_df['contracts'] = trades_df['leg'].map(NOTIONAL_CONTRACTS)

    if trades_df['contracts'].isna().any():
        missing = trades_df.loc[trades_df['contracts'].isna(), 'leg'].unique()
        raise ValueError(f"Unmapped legs found in NOTIONAL_CONTRACTS: {missing}")

    if trades_df['position'].isna().any():
        bad = trades_df.loc[trades_df['position'].isna(), 'leg'].unique()
        raise ValueError(f"Unclassified legs: {bad}")

    cols = [
        'session_start', 'session_type', 'leg', 'strike', 'option_type',
        'position', 'entry_price', 'expiry', 'contracts'
    ]

    trades_df = trades_df[cols]
    return trades_df

In [12]:
def calculate_exit_price_and_pnl(trades_df, btc_prices):
    btc_prices_keyed = btc_prices[['time', 'close']].rename(
        columns={'time': 'expiry', 'close': 'spot_at_expiry'}
    )
    
    trades_df = trades_df.merge(
        btc_prices_keyed,
        on='expiry',
        how='left'
    )

    if trades_df['spot_at_expiry'].isna().any():
        missing = trades_df.loc[trades_df['spot_at_expiry'].isna(), 'expiry'].unique()
        raise ValueError(f"Missing BTC prices for expiries: {missing}")
    
    spot = trades_df['spot_at_expiry']
    strike = trades_df['strike']

    call_payoff = np.maximum(spot - strike, 0)
    put_payoff  = np.maximum(strike - spot, 0)

    trades_df['exit_price'] = np.select(
        [
            trades_df['option_type'] == 'C',
            trades_df['option_type'] == 'P'
        ],
        [
            call_payoff,
            put_payoff
        ],
        default=np.nan
    )

    trades_df['position_sign'] = trades_df['position'].map({'short': -1, 'long': 1})

    trades_df['pnl'] = (
        (trades_df['entry_price'] * trades_df['spot_at_expiry'] - trades_df['exit_price'])
        * trades_df['contracts']
        * trades_df['position_sign']
    )

    trades_df.drop(columns=['position_sign'], inplace=True)

    return trades_df

In [17]:
def summarise_pnl(trades_df):
    trades_df['premium_component'] = np.where(
        trades_df['position'] == 'short',
        trades_df['entry_price'],
        0
    )

    trades_df['hedge_component'] = np.where(
        trades_df['position'] == 'long',
        trades_df['entry_price'],
        0
    )

    agg = trades_df.groupby('session_start').agg(
        gross_pnl_usd=('pnl', 'sum'),
        premium_sum=('premium_component', 'sum'),
        hedge_sum=('hedge_component', 'sum'),
        spot_at_expiry=('spot_at_expiry', 'first')  # assumes constant per session
    )

    agg = agg.sort_values('session_start')

    agg['premiums_received'] = agg['premium_sum'] * agg['spot_at_expiry']
    agg['hedge_cost'] = agg['hedge_sum'] * agg['spot_at_expiry']
    agg['net_entry_premium'] = agg['premiums_received'] - agg['hedge_cost']

    agg['cumulative_pnl_usd'] = agg['gross_pnl_usd'].cumsum()

    pnl_df = agg.reset_index()[[
        'session_start',
        'gross_pnl_usd',
        'premiums_received',
        'hedge_cost',
        'net_entry_premium',
        'cumulative_pnl_usd'
    ]]
    
    return pnl_df

In [14]:
session_signals, options_df, btc_prices = import_data()

In [15]:
trades_df = construct_trades(session_signals, options_df)
trades_df = calculate_exit_price_and_pnl(trades_df, btc_prices)
trades_df

,session_start,session_type,leg,strike,option_type,position,entry_price,expiry,contracts,spot_at_expiry,exit_price,pnl
0,2025-01-13 08:00:00+00:00,weekday,atm_call,93000.0,C,short,0.0110,2025-01-14 00:00:00+00:00,1,94437.17,1437.17,398.361130
1,2025-01-13 08:00:00+00:00,weekday,atm_put,93000.0,P,short,0.0110,2025-01-14 00:00:00+00:00,1,94437.17,0.00,-1038.808870
2,2025-01-13 08:00:00+00:00,weekday,otm_call,100000.0,C,long,0.0001,2025-01-14 00:00:00+00:00,1,94437.17,0.00,9.443717
3,2025-01-13 08:00:00+00:00,weekday,otm_put,84000.0,P,long,0.0002,2025-01-14 00:00:00+00:00,1,94437.17,0.00,18.887434
4,2025-01-20 08:00:00+00:00,weekday,atm_call,108000.0,C,short,0.0215,2025-01-21 00:00:00+00:00,1,101057.33,0.00,-2172.732595
5,2025-01-20 08:00:00+00:00,weekday,atm_put,108000.0,P,short,0.0190,2025-01-21 00:00:00+00:00,1,101057.33,6942.67,5022.580730
6,2025-01-20 08:00:00+00:00,weekday,otm_call,122000.0,C,long,0.0016,2025-01-21 00:00:00+00:00,1,101057.33,0.00,161.691728
7,2025-01-20 08:00:00+00:00,weekday,otm_put,92000.0,P,long,0.0002,2025-01-21 00:00:00+00:00,1,101057.33,0.00,20.211466
8,2025-04-07 08:00:00+00:00,weekday,atm_call,76000.0,C,short,0.0230,2025-04-08 00:00:00+00:00,1,79140.01,3140.01,1319.789770
9,2025-04-07 08:00:00+00:00,weekday,atm_put,75000.0,P,short,0.0115,2025-04-08 00:00:00+00:00,1,79140.01,0.00,-910.110115


In [18]:
pnl_df = summarise_pnl(trades_df)
pnl_df

,session_start,gross_pnl_usd,premiums_received,hedge_cost,net_entry_premium,cumulative_pnl_usd
0,2025-01-13 08:00:00+00:00,-612.116589,2077.617740,28.331151,2049.286589,-612.116589
1,2025-01-20 08:00:00+00:00,3031.751329,4092.821865,181.903194,3910.918671,2419.634740
2,2025-04-07 08:00:00+00:00,504.647667,2730.330345,94.968012,2635.362333,2924.282407
